In [20]:
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import mplhep
import boost_histogram as bh
import glob
from tqdm import tqdm
import numpy as np

In [21]:
# data = uproot.open("build/17230.root")
# histograms = data["histograms"]


In [22]:
# for hist_name, hist in histograms.items():

#     h = hist.to_boost()
#     fig, ax = plt.subplots()
#     mplhep.histplot(h, ax=ax, flow='none', label=hist_name)
#     ax.set_title(hist_name)

#     # underflow = h[bh.underflow]
#     underflow = h[0]
#     print(f"Underflow for {hist_name}: {underflow}")
#     ax.set_yscale('log')

#     print(h)

In [23]:
filelist = sorted(glob.glob("AnalysisZipFramework/submission/analysis_output/*.root"))

h_2022 = None
h_2023 = None
h_2024_preCaloNu = None
h_2024_withCaloNu = None
h_2024_postCaloNu = None

for fpath in tqdm(filelist):
    data = uproot.open(fpath)
    histograms = data["histograms"]
    meta = data["meta"]

    lumi = meta["lumi"].array()[0]
    run_number = int(fpath.split("/")[-1].split(".")[0].split("_")[-1])

    h = histograms["VetoNu0_reduced_charge"].to_boost()

    if len(lumi) == 0:
        continue
    else:
        lumi = lumi[0]

    h = h / lumi    
    if run_number < 1e4:
        print(f"Adding run {run_number} to 2022")
        if h_2022 is None:
            h_2022 = h
        else:
            h_2022 += h
    elif run_number < 1.2e4:
        print(f"Adding run {run_number} to 2023")
        if h_2023 is None:
            h_2023 = h
        else:
            h_2023 += h
    
    elif run_number < 15821: # 2024_preCaloNu
        print(f"Adding run {run_number} to 2024_preCaloNu")
        if h_2024_preCaloNu is None:
            h_2024_preCaloNu = h
        else:
            h_2024_preCaloNu += h
    
    elif 15821 < run_number < 16924: # 2024_CaloNu
        print(f"Adding run {run_number} to 2024_withCaloNu")
        if h_2024_withCaloNu is None:
            h_2024_withCaloNu = h
        else:
            h_2024_withCaloNu += h

    elif run_number > 16924: # 2024_afterCaloNu
        print(f"Adding run {run_number} to 2024_postCaloNu")
        if h_2024_postCaloNu is None:
            h_2024_postCaloNu = h
        else:
            h_2024_postCaloNu += h

  0%|          | 0/555 [00:00<?, ?it/s]


KeyInFileError: not found: 'histograms' (with any cycle number)

    Available keys: 'nt;1', 'meta;1', 'cutflow;1', 'eventID_pass;1'

in file AnalysisZipFramework/submission/analysis_output/10417.root

In [ ]:
fig, ax = plt.subplots()
mplhep.histplot(h_2022, ax=ax, flow='none', label="2022", density=True)
mplhep.histplot(h_2023, ax=ax, flow='none', label="2023", density=True)
mplhep.histplot(h_2024_preCaloNu, ax=ax, flow='none', label="2024 preCaloNu", density=True)
mplhep.histplot(h_2024_withCaloNu, ax=ax, flow='none', label="2024 withCaloNu", density=True)
mplhep.histplot(h_2024_postCaloNu, ax=ax, flow='none', label="2024 postCaloNu", density=True)
ax.set_title("VetoNu0_reduced_charge")
ax.set_yscale('log')
ax.legend()
ax.set_xlim(0, 2000)
ax.set_ylabel("Events (unit normalised)")
ax.set_xlabel("VetoNu0 reduced charge [pC]")
plt.savefig("VetoNu0_reduced_charge.png", dpi=300)
plt.savefig("VetoNu0_reduced_charge.pdf", dpi=300)

In [ ]:
run_numbers = []
n_fail_events = []
n_fail_events_err = []

for fpath in tqdm(filelist):
    data = uproot.open(fpath)
    histograms = data["histograms"]
    meta = data["meta"]

    lumi = meta["lumi"].array()[0]
    run_number = int(fpath.split("/")[-1].split(".")[0].split("_")[-1])

    h = histograms["VetoNu0_reduced_charge"].to_boost()

    if len(lumi) == 0:
        continue
    else:
        lumi = lumi[0]

    if run_number < 1.2e4:
        continue
    
    run_numbers.append(run_number)
    n_fail_events.append(h[0] / lumi)
    n_fail_events_err.append(np.sqrt(h[0]) / lumi)

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(run_numbers, n_fail_events, yerr=n_fail_events_err, fmt='o')
ax.axvline(x=15821, color='r', linestyle='--', label='CaloNu Start')
ax.axvline(x=16924, color='g', linestyle='--', label='CaloNu End')
ax.set_xlabel("Run Number")
ax.set_ylabel("Failed Events / pb")
ax.legend()
ax.set_title("2024 data: VetoNu0_reduced_charge == 0 && VetoNu0_raw_charge > 30", fontsize=8, loc='left')

In [ ]:
files_2024 = [f for f in filelist if int(f.split("/")[-1].split(".")[0].split("_")[-1]) > 1.2e4]
files_caloNu = [f for f in files_2024 if 15821 <= int(f.split("/")[-1].split(".")[0].split("_")[-1]) <= 16924]

In [ ]:
d = uproot.open(files_caloNu[0])["nt"]
for key in d.keys():
    if "calonu" in key.lower():
        print(key)

In [ ]:
h_caloNuEdep = bh.Histogram(bh.axis.Regular(50, 0, 100e3))
for f in tqdm(files_caloNu):
    d = uproot.open(f)["nt"]
    calonuEdep = d["CaloNu_total_E_EM"].array()
    h_caloNuEdep.fill(calonuEdep)

In [ ]:
print(h_caloNuEdep)

In [ ]:
fig, ax = plt.subplots()
mplhep.histplot(h_caloNuEdep, ax=ax, flow='none', label="CaloNu Edep")
ax.set_title("CaloNu Total Edep", fontsize=8, loc='left')
ax.set_xlabel("CaloNu Edep")
ax.set_ylabel("Events")

In [ ]:
mc_caloNu = ["build/200139.root:nt", "build/200140.root:nt", "build/200146.root:nt"]


In [ ]:
h_caloNuEdep_MC = bh.Histogram(bh.axis.Regular(50, 0, 100e3))
h_caloNuEdep_vs_zpos = bh.Histogram(bh.axis.Regular(50, -3100, -1500), bh.axis.Regular(50, 0, 100e3))
for f in tqdm(mc_caloNu):
    d = uproot.open(f)
    
    calonuEdep = d["CaloNu_total_E_EM"].array()
    zpos = d["truth_dec_z"].array()
    h_caloNuEdep_MC.fill(calonuEdep)
    print(calonuEdep)

    h_caloNuEdep_vs_zpos.fill(zpos[:, 0], calonuEdep)

h_caloNuEdep_MC = h_caloNuEdep_MC * 70 / 10_000

In [ ]:
from matplotlib.colors import LogNorm
fig, ax = plt.subplots()
mplhep.hist2dplot(h_caloNuEdep_vs_zpos, ax=ax, flow='show', cmap='viridis', norm=LogNorm())
ax.set_xlabel("Truth neutrino vertex z [mm]")
ax.set_ylabel("CaloNu_total_E_EM [MeV]")


In [ ]:
fig, ax = plt.subplots()
mplhep.histplot(h_caloNuEdep_MC, ax=ax, flow='show', label="MC CaloNu Edep", histtype='fill')
mplhep.histplot(h_caloNuEdep, ax=ax, flow='show', label="Data CaloNu Edep", histtype='errorbar')
ax.set_title("MC CaloNu Total Edep", fontsize=8, loc='left')
ax.set_xlabel("CaloNu Edep")
ax.set_ylabel("Events")
ax.set_yscale('log')

In [ ]:
0 = good hit
1 = below threshold
2 = secondary hit
4 = amplitude overflow
8 = find baseline failed
16 = gaus fit failed
32 = cryst-ball fit failed
64 = invalid clock
128 = waveform missing
256 = waveform invalid


In [ ]:
print(f"MC CaloNu Edep sum (with flow): {h_caloNuEdep_MC.sum(flow=True)}")
print(f"Data CaloNu Edep sum (with flow): {h_caloNuEdep.sum(flow=True)}")

In [24]:
for  f in filelist:
    data = uproot.open(f)["nt"]
    events = data.arrays(["VetoNu0_reduced_charge", "VetoNu0_raw_charge", "VetoNu1_reduced_charge", "VetoNu1_raw_charge", "run", "VetoNu0_status", "VetoNu1_status",
                          "Veto20_status", "Veto21_status", "Preshower0_status", "Preshower1_status"
                          ]
                         , library="pd")
    print(f"Nevents = {len(events)}")
    events = events.query("(VetoNu0_reduced_charge == 0 and VetoNu0_raw_charge > 30) or (VetoNu1_reduced_charge == 0 and VetoNu1_raw_charge > 30)")
    print(len(events))
    if len(events) > 0:
        display(events)

    

Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 4
0
Nevents = 2
0
Nevents = 1
0
Nevents = 0
0
Nevents = 1
0
Nevents = 1
0
Nevents = 0
0
Nevents = 3
0
Nevents = 2
0
Nevents = 4
1


,VetoNu0_reduced_charge,VetoNu0_raw_charge,VetoNu1_reduced_charge,VetoNu1_raw_charge,run,VetoNu0_status,VetoNu1_status,Veto20_status,Veto21_status,Preshower0_status,Preshower1_status
2,0.0,94.840775,0.0,193.076172,10701,0,0,4,4,0,0


Nevents = 3
0
Nevents = 2
0
Nevents = 2
0
Nevents = 1
0
Nevents = 0
0
Nevents = 0
0
Nevents = 2
0
Nevents = 6
0
Nevents = 1
0
Nevents = 2
0
Nevents = 0
0
Nevents = 2
0
Nevents = 3
0
Nevents = 4
0
Nevents = 2
0
Nevents = 1
0
Nevents = 4
1


,VetoNu0_reduced_charge,VetoNu0_raw_charge,VetoNu1_reduced_charge,VetoNu1_raw_charge,run,VetoNu0_status,VetoNu1_status,Veto20_status,Veto21_status,Preshower0_status,Preshower1_status
1,0.0,109.530174,0.0,92.85994,10881,0,0,4,4,0,0


Nevents = 0
0
Nevents = 0
0
Nevents = 2
0
Nevents = 3
0
Nevents = 3
0
Nevents = 8
0
Nevents = 0
0
Nevents = 1
0
Nevents = 0
0
Nevents = 3
1


...

Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 1
0
Nevents = 1
0
Nevents = 2
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 1
0
Nevents = 2
0
Nevents = 1
0
Nevents = 0
0
Nevents = 1
0
Nevents = 2
0
Nevents = 6
0
Nevents = 1
0
Nevents = 2
0
Nevents = 1
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 1
0
Nevents = 1
0
Nevents = 2
0
Nevents = 0
0
Nevents = 2
0
Nevents = 7
0
Nevents = 0
0
Nevents = 2
0
Nevents = 3
0
Nevents = 4
0
Nevents = 2
0
Nevents = 0
0
Nevents = 4
0
Nevents = 1
0
Nevents = 3
0
Nevents = 0
0
Nevents = 8
0
Nevents = 1
0
Nevents = 4
0
Nevents = 3
0
Nevents = 5
0
Nevents = 5
0
Nevents = 2
0
Nevents = 6
0
Nevents = 5
0
Nevents = 2
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 1
0
Nevents = 3
0
Nevents = 1
0
Nevents = 0
0
Nevents = 4
0
Nevent

,VetoNu0_reduced_charge,VetoNu0_raw_charge,VetoNu1_reduced_charge,VetoNu1_raw_charge,run,VetoNu0_status,VetoNu1_status,Veto20_status,Veto21_status,Preshower0_status,Preshower1_status
2,0.0,33.42337,0.136074,0.687852,8733,0,1,4,4,0,0


Nevents = 0
0
Nevents = 1
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 2
0
Nevents = 0
0
Nevents = 1
0
Nevents = 2
0
Nevents = 0
0
Nevents = 5
0
Nevents = 0
0
Nevents = 1
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 1
0
Nevents = 5
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 6
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 2
0
Nevents = 3
0
Nevents = 3
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 0
0
Nevents = 2
0
Nevents = 1
0
Nevents = 5
0
Nevents = 0
0
Nevents = 4
0
Nevents = 1
0
Nevents = 3
0
Nevents = 9
0
Nevents = 1
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 4
0
Nevents = 3
0
Nevents = 1
0
Nevents = 3
0
Nevents = 1
0
Nevents = 3
0
Nevents = 0
0
Nevents = 0
0
Nevents = 0
0
Nevents = 1
0
Nevents = 2
0
Nevents = 0
0
Nevents = 0
0
Nevents = 4
1


,VetoNu0_reduced_charge,VetoNu0_raw_charge,VetoNu1_reduced_charge,VetoNu1_raw_charge,run,VetoNu0_status,VetoNu1_status,Veto20_status,Veto21_status,Preshower0_status,Preshower1_status
0,0.408965,2.092144,0.0,116.051506,9069,0,0,4,4,0,0


Nevents = 6
0
Nevents = 6
0
Nevents = 5
0
Nevents = 3
0
Nevents = 0
0
Nevents = 0
0
Nevents = 3
0
Nevents = 0
0
Nevents = 2
0
Nevents = 0
0
Nevents = 0
0
Nevents = 5
0
Nevents = 4
0
Nevents = 1
0
